In [ ]:
import pandas as pd

# 1. Učitavanje podataka iz Excela
file_name = r"D:\Projects\EBITDA simulation\finance.xlsx"

# Čitamo Income Statement
df_is = pd.read_excel(file_name, sheet_name='Income_statement', skiprows=1)

# Sređujemo indeks
df_is.dropna(subset=[df_is.columns[0]], inplace=True)
df_is.set_index(df_is.columns[0], inplace=True)

# Osiguravamo format kolona (za svaki slučaj ako su godine u Excelu brojevi)
df_is.columns = df_is.columns.astype(str)

# Uzimamo podatke iz poslednje istorijske godine (2025) kao bazu za projekciju
rev_2025 = df_is.loc['Revenue', '2025']
cogs_2025 = df_is.loc['COGS', '2025']
sga_2025 = df_is.loc['SG&A', '2025']
depr_2025 = df_is.loc['Depreciation', '2025']
interest_2025 = df_is.loc['Interest', '2025']

# PROJEKCIJA BAZNE 2026. GODINE (Pre optimizacije)
# Pretpostavljamo realan rast prihoda od npr. 12% u odnosu na 2025. godinu
stopa_rasta = 0.12 
revenue_2026 = rev_2025 * (1 + stopa_rasta)



# Zadržavamo istu istorijsku strukturu troškova (% od prihoda) za baznu 2026. godinu
cogs_procenat = cogs_2025 / rev_2025
sga_procenat = sga_2025 / rev_2025

cogs_2026_bazni = revenue_2026 * cogs_procenat
sga_2026_bazni = revenue_2026 * sga_procenat

# Fiksni troškovi prate 2025. godinu
depreciation_2026 = depr_2025
interest_2026 = interest_2025

# Trenutna biznis EBITDA za 2026. pre nego što unesemo novi cilj
trenutna_ebitda_2026 = revenue_2026 + cogs_2026_bazni + sga_2026_bazni
trenutna_margina_2026 = (trenutna_ebitda_2026 / revenue_2026) * 100

print("--- PROJEKCIJA ZA BAZNU 2026. GODINU (Bez optimizacije) ---")
print(f"Projektovani Prihod (Revenue): ${revenue_2026:,.2f}")
print(f"Bazna EBITDA: ${trenutna_ebitda_2026:,.2f} (Margina: {trenutna_margina_2026:.2f}%)")
print("\n" + "="*55 + "\n")

# 2. Unos željene EBITDA margine za 2026. godinu
try:
    target_margin_input = float(input("Unesite željenu EBITDA marginu za 2026. godinu u % (npr. 23.5): "))
    target_margin = target_margin_input / 100
except ValueError:
    print("Molimo unesite validan broj.")
    exit()

# 3. Pokretanje simulacije SAMO ZA 2026. GODINU
target_ebitda_2026 = revenue_2026 * target_margin

# Ukupni dozvoljeni operativni troškovi za ciljanu EBITDA
dozvoljeni_troškovi = revenue_2026 - target_ebitda_2026

# Proporcija između COGS i SG&A na osnovu istorijskih podataka iz 2025. godine
ukupni_opex_2025 = abs(cogs_2025) + abs(sga_2025)
proporcija_cogs = abs(cogs_2025) / ukupni_opex_2025
proporcija_sga = abs(sga_2025) / ukupni_opex_2025

# Novi optimizovani COGS i SG&A za 2026. godinu
novi_cogs_2026 = -(dozvoljeni_troškovi * proporcija_cogs)
novi_sga_2026 = -(dozvoljeni_troškovi * proporcija_sga)

# Izračunavanje potrebnih procenata uštede u odnosu na bazni plan za 2026.
potrebna_usteda_cogs = (1 - (abs(novi_cogs_2026) / abs(cogs_2026_bazni))) * 100
potrebna_usteda_sga = (1 - (abs(novi_sga_2026) / abs(sga_2026_bazni))) * 100

# Rekalkulacija celog Bilansa uspeha za 2026. godinu
novi_gross_profit = revenue_2026 + novi_cogs_2026
novi_ebt = novi_gross_profit + depreciation_2026 + novi_sga_2026 + interest_2026
novi_tax = novi_ebt * 0.20 if novi_ebt > 0 else 0
novi_net_earnings = novi_ebt - novi_tax

# Pakovanje rezultata u DataFrame koji prikazuje samo 2026. godinu
simulacija_2026 = {
    'Finansijska Stavka': [
        '1. Projektovani Prihod (Revenue)',
        '2. Potreban Optimizovani COGS',
        '3. POTREBNA UŠTEDA NA COGS (%)',
        '4. Potreban Optimizovani SG&A',
        '5. POTREBNA UŠTEDA NA SG&A (%)',
        '6. CILJANA EBITDA VREDNOST',
        '7. Novi Neto Dobitak (Net Earnings)'
    ],
    'Projekcija za 2026. godinu': [
        f"${revenue_2026:,.2f}",
        f"${novi_cogs_2026:,.2f}",
        f"{potrebna_usteda_cogs:.2f}%",
        f"${novi_sga_2026:,.2f}",
        f"{potrebna_usteda_sga:.2f}%",
        f"${target_ebitda_2026:,.2f}",
        f"${novi_net_earnings:,.2f}"
    ]
}

df_rezultat = pd.DataFrame(simulacija_2026)
df_rezultat.set_index('Finansijska Stavka', inplace=True)

print(f"\n--- STRATEŠKI PLAN OPTIMIZACIJE ZA 2026. GODINU (Cilj: {target_margin_input}%) ---")
print(df_rezultat)

--- PROJEKCIJA ZA BAZNU 2026. GODINU (Bez optimizacije) ---
Projektovani Prihod (Revenue): $24,640.00
Bazna EBITDA: $4,928.00 (Margina: 20.00%)



--- STRATEŠKI PLAN OPTIMIZACIJE ZA 2026. GODINU (Cilj: 23.5%) ---
                                    Projekcija za 2026. godinu
Finansijska Stavka                                            
1. Projektovani Prihod (Revenue)                    $24,640.00
2. Potreban Optimizovani COGS                      $-14,137.20
3. POTREBNA UŠTEDA NA COGS (%)                           4.38%
4. Potreban Optimizovani SG&A                       $-4,712.40
5. POTREBNA UŠTEDA NA SG&A (%)                           4.38%
6. CILJANA EBITDA VREDNOST                           $5,790.40
7. Novi Neto Dobitak (Net Earnings)                  $4,032.32
